# Mental Health RAG model using Gemini and CHROMADB


> [Credit to Kaggle notebook](https://www.kaggle.com/code/markishere/day-2-document-q-a-with-rag)



In [1]:
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 6.8 MB/s eta 0:0

In [70]:
from google import genai
from google.genai import types
import requests
from bs4 import BeautifulSoup

import os

from IPython.display import Markdown

genai.__version__

'1.7.0'

In [3]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('API_KEY') # you need to set yours

In [4]:
# List of models
client = genai.Client(api_key=GOOGLE_API_KEY)

for m in client.models.list():
  if "embedContent" in m.supported_actions:
    print(m.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp


We will use the text-embedding-004 model

# Function to red an url and returns a list of paragraphs

In [5]:
# read an html file from a url
def read_html_from_url(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    # return soup.get_text()
    paragraphs = soup.find_all('p')
    paragraph_texts = [p.get_text() for p in paragraphs]
    return paragraph_texts



# **Caution** : This code can be blocked if executed many times !

In [7]:
doc1 = read_html_from_url("https://www.nychealthandhospitals.org/healthtips/7-tips-to-avoid-stress/")

In [9]:
len(doc1)

4

In [10]:
documents = [doc for doc in doc1]

In [14]:
documents[0]

'There are emotional and behavioral consequences of stress that can make it difficult to perform your daily routine. These include anxiety, depression, fatigue, becoming aggressive, unmotivated, or withdrawn, and difficulties with problem-solving and concentration. There are also physiological consequences of stress including headaches, nausea, and palpitations.\nSo, how can you avoid stress and assuage the negative social, emotional and physical consequences of it in the process?\nHere are some tips:'

# So I will use local html files

# Function to read a local .html file ans returns a list of paragraphs

In [16]:
# read local html file
def read_html_from_file(file_path):
  with open(file_path, 'r') as file:
    content = file.read()
    soup = BeautifulSoup(content, 'html.parser')
    paragraphs = soup.find_all('p')
    paragraph_texts = [p.get_text() for p in paragraphs]
    return paragraph_texts

In [ ]:
documents2 = read_html_from_file("How_to_Identify_Stress.html") # change to your file path

In [26]:
len(documents2)

31

In [29]:
documents2[0]

'Do you often find yourself amidst your thoughts and stress about your current life situation? We often say that we are stressed out about this or that. Do you ever wonder what stress is exactly, how we can identify Stress, and how to manage it successfully?'

In [30]:
documents2[10]

'Stress and the response it creates are important. It might help you forestall fear or distress in situations so you may run a race or pass an exam. Once the event of stress is over, the hormones will go back to their normal levels without any lasting effects.'

In [31]:
documents2[20]

'Start a Fundraiser on Ketto and raise the amount for your treatment'

In [32]:
documents2[30]

'Neve | Powered by Powered by WordPress.com.'

In [33]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry
from google.genai import types

In [34]:
# Define a helper to retry when per-minute quota is reached.
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

In [35]:
class GeminiEmbeddingFunction(EmbeddingFunction):
  # Specify whether to generate embeddings for documents, or queries
  document_mode = True

  @retry.Retry(predicate=is_retriable)
  def __call__(self, input: Documents) -> Embeddings:
    if self.document_mode:
      embedding_task = "retrieval_document"
    else:
      embedding_task = "retrieval_query"

    response = client.models.embed_content(
            model="models/text-embedding-004",
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
    return [e.values for e in response.embeddings]

In [52]:
import chromadb

DB_NAME = "mental_health_docs2" # to change

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents2, ids=[str(i) for i in range(len(documents2))])

In [53]:
db.count()

31

In [54]:
db.peek(10)#['embeddings'].shape

{'ids': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
 'embeddings': array([[ 0.03544145, -0.01555961, -0.02791646, ..., -0.01859349,
          0.03326548, -0.06773808],
        [ 0.04396529, -0.0179666 , -0.03638542, ..., -0.02177081,
          0.04309296, -0.06891783],
        [ 0.02983765, -0.00380692, -0.02968752, ..., -0.04316411,
          0.04982437, -0.06675615],
        ...,
        [ 0.03833747,  0.00959192, -0.02444811, ..., -0.04811049,
          0.01006862, -0.06178212],
        [ 0.04592604,  0.00062671, -0.0219783 , ..., -0.01211151,
          0.03959532, -0.07392502],
        [ 0.03331509, -0.01231733, -0.04796987, ..., -0.0031699 ,
          0.0282465 , -0.07366461]]),
 'documents': ['Do you often find yourself amidst your thoughts and stress about your current life situation? We often say that we are stressed out about this or that. Do you ever wonder what stress is exactly, how we can identify Stress, and how to manage it successfully?',
  'Don’t worry. This ar

# Retrieval: Find relevant documents

In [55]:
# Switch to query mode when generating embeddings.
embed_fn.document_mode = False  # remember that embed_fn = GeminiEmbeddingFunction()

The initial step to managing stress is to understand the after-effects of stress. Be that as it may, perceiving tension indications may not be as easy as you think. The majority of us are so used to being distressed, identifying stress is difficult. We never realize we are stressed until it’s too late.

In [66]:
# Search the Chroma DB using the specified query.
query = "I think I have stress. What can I do to fill better ?"

result = db.query(query_texts=[query], n_results=5)
[all_passages] = result["documents"]

In [67]:
for passage in all_passages:
  print(passage)
  print("-------------------------------------------------------------------")

There are times when we need help to Identify stress. Not just a family member but like a doctor or a counsellor. This is not an indication of shortcoming but an acknowledgement that we need to practice new things when faced with new conditions. Know that you are not alone. Many others might be battling something similar or the same. Sharing these thoughts might help you in many ways. You can join support groups according to your issues. These are anonymous, and hence you need not worry about venting things out.
-------------------------------------------------------------------
Don’t worry. This article is going to guide you through every aspect of stress, help you to identify stress and overcome it.
-------------------------------------------------------------------
It’s better to look for the root cause of your stress and learn how to manage it.
-------------------------------------------------------------------
Do you often find yourself amidst your thoughts and stress about your c

# Augmented generation: Answer the question

In [68]:
query_oneline = query.replace("\n", " ")

# This prompt is where you can specify any guidance on tone, or what topics the model should stick to, or avoid.
prompt = f"""You are a psychoanalyst that answers questions using text from
the reference passage included below.

QUESTION: {query_oneline}
"""

# Add the retrieved documents to the prompt.
for passage in all_passages:
    passage_oneline = passage.replace("\n", " ")
    prompt += f"PASSAGE: {passage_oneline}\n"

print(prompt)

You are a psychoanalyst that answers questions using text from 
the reference passage included below.

QUESTION: I think I have stress. What can I do to fill better ?
PASSAGE: There are times when we need help to Identify stress. Not just a family member but like a doctor or a counsellor. This is not an indication of shortcoming but an acknowledgement that we need to practice new things when faced with new conditions. Know that you are not alone. Many others might be battling something similar or the same. Sharing these thoughts might help you in many ways. You can join support groups according to your issues. These are anonymous, and hence you need not worry about venting things out.
PASSAGE: Don’t worry. This article is going to guide you through every aspect of stress, help you to identify stress and overcome it.
PASSAGE: It’s better to look for the root cause of your stress and learn how to manage it.
PASSAGE: Do you often find yourself amidst your thoughts and stress about your cu

In [69]:
answer = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt)

Markdown(answer.text)

It's important to identify the root cause of your stress and learn how to manage it. If needed, seek help from a doctor or counselor to identify your stress. Sharing your thoughts can also be helpful, and you might consider joining a support group. Remember, you're not alone in this.
